# 🛡️ AI-Based Intrusion Detection & Automated Response System (IDRS)
## For Educational Web Platforms — Tunisia Context
---
## PART 1: Environment Setup, Dataset Loading & Exploratory Data Analysis

**Author:** AI Security Engineering Pipeline  
**Target:** Educational institutions (universities, LMS platforms, student portals, e-learning systems)  
**Scope of Part 1:**
- ✅ Full environment setup with version-pinned dependencies
- ✅ Dataset acquisition helpers (CIC-IDS2017 + UNSW-NB15)
- ✅ Deep Exploratory Data Analysis (EDA)
- ✅ Feature engineering for network flow data
- ✅ Data quality audit pipeline
- ✅ Educational web platform threat taxonomy mapping

---
### 📚 Dataset References
| Dataset | Source | Download |
|---|---|---|
| **CIC-IDS2017** | Canadian Institute for Cybersecurity | https://www.unb.ca/cic/datasets/ids-2017.html |
| **UNSW-NB15** | UNSW Canberra Cyber | https://research.unsw.edu.au/projects/unsw-nb15-dataset |
| **SQLi Payloads** | Kaggle | https://www.kaggle.com/datasets/sajid576/sql-injection-dataset |

> **Integration Note:** Download CIC-IDS2017 CSV files and place them in `./data/cicids2017/`.  
> Download UNSW-NB15 CSV files and place them in `./data/unsw_nb15/`.  
> The cells below will automatically detect and load available files.

---
## 📦 CELL 1 — Install All Required Libraries

In [ ]:
# ============================================================
# IDRS — Dependency Installation
# Run once. Restart kernel after completion.
# ============================================================
import subprocess, sys

packages = [
    # Core data science
    "numpy==1.26.4",
    "pandas==2.2.2",
    "scipy==1.13.0",

    # Visualization
    "matplotlib==3.8.4",
    "seaborn==0.13.2",
    "plotly==5.22.0",
    "kaleido==0.2.1",          # static plotly export

    # Machine learning
    "scikit-learn==1.4.2",
    "xgboost==2.0.3",
    "lightgbm==4.3.0",
    "imbalanced-learn==0.12.2",  # SMOTE
    "optuna==3.6.1",             # Bayesian hyperparameter tuning
    "shap==0.45.0",              # Explainability
    "joblib==1.4.0",

    # Deep learning
    "torch==2.3.0",
    "torchvision==0.18.0",

    # NLP / LLM pipeline
    "transformers==4.41.0",
    "datasets==2.19.1",
    "tokenizers==0.19.1",
    "accelerate==0.30.1",
    "huggingface_hub==0.23.0",
    "evaluate==0.4.2",

    # Online / streaming ML
    "river==0.21.0",             # concept drift, online learning

    # Utilities
    "tqdm==4.66.4",
    "colorama==0.4.6",
    "tabulate==0.9.0",
    "ipywidgets==8.1.2",
    "kaggle==1.6.14",            # dataset download helper
    "requests==2.31.0",
]

print("📦 Installing IDRS dependencies...")
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  {status} {pkg}")

print("\n🔁 All packages installed. Restart the kernel before proceeding.")

---
## ⚙️ CELL 2 — Global Imports & Configuration

In [ ]:
# ============================================================
# IDRS — Global Imports & Reproducibility Config
# ============================================================
import os
import sys
import warnings
import random
import json
import hashlib
import logging
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import scipy.stats as stats

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight

from tqdm import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_colwidth', 40)

# ── Reproducibility seed ──────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# ── Directory structure ───────────────────────────────────
BASE_DIR   = Path(".")
DATA_DIR   = BASE_DIR / "data"
CIC_DIR    = DATA_DIR / "cicids2017"
UNSW_DIR   = DATA_DIR / "unsw_nb15"
SQLI_DIR   = DATA_DIR / "sqli"
OUTPUT_DIR = BASE_DIR / "outputs"
MODEL_DIR  = BASE_DIR / "models"
LOG_DIR    = BASE_DIR / "logs"

for d in [DATA_DIR, CIC_DIR, UNSW_DIR, SQLI_DIR, OUTPUT_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Logging ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(LOG_DIR / "idrs_part1.log"),
        logging.StreamHandler(sys.stdout)
    ]
)
logger = logging.getLogger("IDRS")

# ── Plot style ────────────────────────────────────────────
PALETTE = {
    'benign'    : '#2ecc71',
    'dos'       : '#e74c3c',
    'ddos'      : '#c0392b',
    'probe'     : '#f39c12',
    'web'       : '#9b59b6',
    'brute'     : '#3498db',
    'botnet'    : '#1abc9c',
    'exploit'   : '#e67e22',
    'other'     : '#95a5a6',
}

plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#f8f9fa',
    'axes.grid'        : True,
    'grid.alpha'       : 0.4,
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.labelsize'   : 11,
    'figure.dpi'       : 120,
})

logger.info("✅ Environment configured — SEED=%d, BASE_DIR=%s", SEED, BASE_DIR.resolve())
print(f"\n📂 Directory structure created under: {BASE_DIR.resolve()}")
print(f"   • data/cicids2017/   ← place CIC-IDS2017 CSV files here")
print(f"   • data/unsw_nb15/    ← place UNSW-NB15 CSV files here")
print(f"   • data/sqli/         ← place SQLi payload CSV here")

---
## 🗺️ CELL 3 — Educational Web Platform Threat Taxonomy

Before loading data, we define a canonical threat taxonomy specifically tailored to Tunisian educational platforms (university portals, Moodle-based LMS, student authentication systems, research databases).

In [ ]:
# ============================================================
# IDRS — Threat Taxonomy for Educational Web Platforms
# ============================================================

THREAT_TAXONOMY = {
    # ── Tier 0: No threat ──────────────────────────────────
    "BENIGN": {
        "label_id"    : 0,
        "severity"    : "none",
        "color"       : PALETTE['benign'],
        "edu_context" : "Normal student/staff browsing, LMS access, file downloads",
        "cic_labels"  : ["BENIGN"],
        "unsw_labels" : ["Normal"],
        "response"    : "none",
    },

    # ── Tier 1: Availability attacks ──────────────────────
    "DoS": {
        "label_id"    : 1,
        "severity"    : "high",
        "color"       : PALETTE['dos'],
        "edu_context" : "Taking down exam portals, LMS, or registration systems during peak periods",
        "cic_labels"  : ["DoS slowloris", "DoS Slowhttptest", "DoS Hulk", "DoS GoldenEye",
                         "Heartbleed"],
        "unsw_labels" : ["DoS"],
        "response"    : "rate_limit + block_ip + alert_admin",
    },
    "DDoS": {
        "label_id"    : 2,
        "severity"    : "critical",
        "color"       : PALETTE['ddos'],
        "edu_context" : "Coordinated attack against university main portal or enrollment system",
        "cic_labels"  : ["DDoS"],
        "unsw_labels" : ["DoS"],
        "response"    : "block_subnet + cdn_reroute + emergency_alert",
    },

    # ── Tier 2: Reconnaissance ────────────────────────────
    "Probe": {
        "label_id"    : 3,
        "severity"    : "medium",
        "color"       : PALETTE['probe'],
        "edu_context" : "Port scanning university networks, probing student information systems",
        "cic_labels"  : ["PortScan"],
        "unsw_labels" : ["Reconnaissance", "Fuzzers"],
        "response"    : "log_alert + temp_block + honeypot_redirect",
    },

    # ── Tier 3: Web application attacks ───────────────────
    "WebAttack": {
        "label_id"    : 4,
        "severity"    : "critical",
        "color"       : PALETTE['web'],
        "edu_context" : "SQL injection on student DB, XSS on LMS, CSRF on grade submission forms",
        "cic_labels"  : ["Web Attack - Brute Force", "Web Attack - XSS",
                         "Web Attack - Sql Injection"],
        "unsw_labels" : ["Exploits", "Generic"],
        "response"    : "block_ip + sanitize_input + waf_rule + notify_dba",
    },

    # ── Tier 4: Authentication attacks ────────────────────
    "BruteForce": {
        "label_id"    : 5,
        "severity"    : "high",
        "color"       : PALETTE['brute'],
        "edu_context" : "Password brute-force on student/staff login portals, SSH on servers",
        "cic_labels"  : ["FTP-Patator", "SSH-Patator"],
        "unsw_labels" : ["Generic"],
        "response"    : "account_lockout + captcha_enforce + notify_user",
    },

    # ── Tier 5: Persistent threats ────────────────────────
    "Botnet": {
        "label_id"    : 6,
        "severity"    : "critical",
        "color"       : PALETTE['botnet'],
        "edu_context" : "Compromised student machines used for C&C, data exfiltration",
        "cic_labels"  : ["Bot"],
        "unsw_labels" : ["Backdoors"],
        "response"    : "isolate_host + full_packet_capture + incident_response",
    },

    # ── Tier 6: Exploitation ──────────────────────────────
    "Exploit": {
        "label_id"    : 7,
        "severity"    : "critical",
        "color"       : PALETTE['exploit'],
        "edu_context" : "CVE exploitation on unpatched university servers, shellcode injection",
        "cic_labels"  : ["Infiltration"],
        "unsw_labels" : ["Exploits", "Shellcode", "Worms"],
        "response"    : "emergency_patch_alert + isolate_service + forensic_snapshot",
    },
}

# ── Build flat label → taxonomy lookups ──────────────────
CIC_LABEL_MAP  = {}
UNSW_LABEL_MAP = {}

for threat, info in THREAT_TAXONOMY.items():
    for lbl in info['cic_labels']:
        CIC_LABEL_MAP[lbl.strip().lower()] = threat
    for lbl in info['unsw_labels']:
        UNSW_LABEL_MAP[lbl.strip().lower()] = threat

# ── Visualize taxonomy ────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.axis('off')

table_data = [
    [t, i['label_id'], i['severity'].upper(), i['edu_context'][:65]+"..."]
    for t, i in THREAT_TAXONOMY.items()
]
col_labels = ['Threat Class', 'ID', 'Severity', 'Educational Platform Context']
tbl = ax.table(
    cellText  = table_data,
    colLabels = col_labels,
    cellLoc   = 'left',
    loc       = 'center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.6)

severity_colors = {'NONE':'#d5f5e3','MEDIUM':'#fef9e7','HIGH':'#fdebd0','CRITICAL':'#fadbd8'}
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif col == 2 and row > 0:
        cell.set_facecolor(severity_colors.get(table_data[row-1][2], 'white'))

plt.title('IDRS Threat Taxonomy — Educational Web Platforms', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'threat_taxonomy.png', bbox_inches='tight', dpi=150)
plt.show()

logger.info("Threat taxonomy defined: %d classes, CIC mappings: %d, UNSW mappings: %d",
            len(THREAT_TAXONOMY), len(CIC_LABEL_MAP), len(UNSW_LABEL_MAP))
print(f"\n📌 Taxonomy: {len(THREAT_TAXONOMY)} threat classes | "
      f"{len(CIC_LABEL_MAP)} CIC labels | {len(UNSW_LABEL_MAP)} UNSW labels mapped")

---
## 📥 CELL 4 — Dataset Loading Engine

In [ ]:
# ============================================================
# IDRS — Robust Dataset Loader
# Handles both CIC-IDS2017 and UNSW-NB15 formats
# ============================================================

# ── CIC-IDS2017 column name normalizer ───────────────────
def normalize_cic_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize column names from CIC-IDS2017 (strip whitespace, fix label col)."""
    df.columns = df.columns.str.strip()
    # The label column varies by file — standardize to 'label'
    for candidate in ['Label', 'label', ' Label', 'LABEL']:
        if candidate in df.columns:
            df = df.rename(columns={candidate: 'label'})
            break
    return df


def load_cicids2017(data_dir: Path, sample_frac: float = 1.0,
                    max_rows_per_file: int = 200_000) -> pd.DataFrame:
    """
    Load CIC-IDS2017 from a directory of CSV files.
    
    Parameters
    ----------
    data_dir        : directory containing Monday.csv, Tuesday.csv, ... files
    sample_frac     : fraction to sample per file (1.0 = all)
    max_rows_per_file: cap per CSV to avoid OOM on large machines
    """
    csv_files = sorted(data_dir.glob("*.csv"))
    if not csv_files:
        logger.warning("No CIC-IDS2017 CSVs found in %s — using synthetic demo data.", data_dir)
        return None

    frames = []
    for f in tqdm(csv_files, desc="Loading CIC-IDS2017"):
        try:
            chunk = pd.read_csv(
                f,
                nrows       = max_rows_per_file,
                low_memory  = False,
                encoding    = 'utf-8',
                on_bad_lines= 'skip',
            )
            chunk = normalize_cic_columns(chunk)
            chunk['source_file'] = f.stem
            if 'label' not in chunk.columns:
                logger.warning("Skipping %s — no label column found.", f.name)
                continue
            if sample_frac < 1.0:
                chunk = chunk.sample(frac=sample_frac, random_state=SEED)
            frames.append(chunk)
            logger.info("  Loaded: %-35s  rows=%d", f.name, len(chunk))
        except Exception as e:
            logger.error("  Error loading %s: %s", f.name, str(e))

    if not frames:
        return None

    df = pd.concat(frames, ignore_index=True)
    # Map original labels → taxonomy
    df['threat_class'] = df['label'].str.strip().str.lower().map(CIC_LABEL_MAP).fillna('UNKNOWN')
    df['dataset']      = 'CICIDS2017'
    logger.info("CIC-IDS2017 loaded: %d rows, %d columns", *df.shape)
    return df


def load_unsw_nb15(data_dir: Path, sample_frac: float = 1.0,
                   max_rows_per_file: int = 200_000) -> pd.DataFrame:
    """
    Load UNSW-NB15 from a directory of CSV files.
    Expected files: UNSW_NB15_training-set.csv / UNSW_NB15_testing-set.csv
    or the 4-part UNSW-NB15_1.csv through _4.csv
    """
    csv_files = sorted(data_dir.glob("*.csv"))
    if not csv_files:
        logger.warning("No UNSW-NB15 CSVs found in %s — using synthetic demo data.", data_dir)
        return None

    # UNSW-NB15 feature names (45 + label + attack_cat)
    UNSW_COLS = [
        'srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
        'sttl','dttl','sloss','dloss','service','Sload','Dload','Spkts','Dpkts',
        'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth','res_bdy_len',
        'Sjit','Djit','Stime','Ltime','Sintpkt','Dintpkt','tcprtt','synack','ackdat',
        'is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd','is_ftp_login',
        'ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ltm',
        'ct_src_dport_ltm','ct_dst_sport_ltm','ct_dst_src_ltm','attack_cat','label'
    ]

    frames = []
    for f in tqdm(csv_files, desc="Loading UNSW-NB15"):
        try:
            chunk = pd.read_csv(
                f,
                nrows       = max_rows_per_file,
                low_memory  = False,
                encoding    = 'utf-8',
                on_bad_lines= 'skip',
                header      = 0,
            )
            # If headerless file, assign columns
            if len(chunk.columns) == len(UNSW_COLS) and chunk.columns[0] not in ['srcip','proto']:
                chunk.columns = UNSW_COLS

            chunk['source_file'] = f.stem
            if sample_frac < 1.0:
                chunk = chunk.sample(frac=sample_frac, random_state=SEED)
            frames.append(chunk)
            logger.info("  Loaded: %-35s  rows=%d", f.name, len(chunk))
        except Exception as e:
            logger.error("  Error loading %s: %s", f.name, str(e))

    if not frames:
        return None

    df = pd.concat(frames, ignore_index=True)

    # Normalize label columns
    if 'attack_cat' in df.columns:
        df['attack_cat'] = df['attack_cat'].fillna('Normal').str.strip()
    if 'label' not in df.columns and 'Label' in df.columns:
        df = df.rename(columns={'Label': 'label'})

    # Map to taxonomy
    label_src = 'attack_cat' if 'attack_cat' in df.columns else 'label'
    df['threat_class'] = df[label_src].str.strip().str.lower().map(UNSW_LABEL_MAP).fillna('UNKNOWN')
    df['dataset']      = 'UNSW_NB15'
    logger.info("UNSW-NB15 loaded: %d rows, %d columns", *df.shape)
    return df


print("✅ Dataset loader functions defined.")
print("   → load_cicids2017(CIC_DIR)")
print("   → load_unsw_nb15(UNSW_DIR)")

---
## 🎭 CELL 5 — Synthetic Demo Generator

If you have not yet downloaded the datasets, this cell generates a **statistically realistic synthetic dataset** that mirrors the CIC-IDS2017 feature distributions. This allows the full notebook to run end-to-end for testing purposes.

In [ ]:
# ============================================================
# IDRS — Synthetic Data Generator (Demo / CI mode)
# Mirrors CIC-IDS2017 statistical distributions per class
# ============================================================

def generate_synthetic_idrs_data(n_samples: int = 80_000,
                                  seed: int = SEED) -> pd.DataFrame:
    """
    Generate a synthetic network intrusion dataset that faithfully
    approximates CIC-IDS2017 feature statistics per attack class.
    
    Features generated (subset of CIC-IDS2017 78 features):
        Flow duration, packet counts, byte counts, inter-arrival times,
        flag counts, packet length statistics, flow rates.
    """
    rng = np.random.default_rng(seed)

    # ── Per-class statistical profiles ──────────────────────
    # Format: (mean_duration, mean_pkt_len, mean_fwd_pkts, mean_bwd_pkts,
    #          mean_fwd_bytes, mean_bwd_bytes, syn_flag_mean, rst_flag_mean,
    #          flow_bytes_s_mean, n_proportional)
    CLASS_PROFILES = {
        'BENIGN'    : dict(dur_mean=5.0,  pkt_mean=800,  fwd_pkts=12, bwd_pkts=10,
                           fwd_bytes=8000, bwd_bytes=12000, syn=0.1, rst=0.02,
                           flow_bps=3000,  weight=0.55),
        'DoS'       : dict(dur_mean=0.001,pkt_mean=60,   fwd_pkts=20, bwd_pkts=1,
                           fwd_bytes=1200, bwd_bytes=100,  syn=0.8, rst=0.3,
                           flow_bps=1e6,   weight=0.12),
        'DDoS'      : dict(dur_mean=0.0001,pkt_mean=50,  fwd_pkts=50, bwd_pkts=0,
                           fwd_bytes=2500, bwd_bytes=0,    syn=1.0, rst=0.1,
                           flow_bps=2e6,   weight=0.08),
        'Probe'     : dict(dur_mean=0.01, pkt_mean=100,  fwd_pkts=3,  bwd_pkts=2,
                           fwd_bytes=300,  bwd_bytes=200,  syn=0.6, rst=0.4,
                           flow_bps=50000, weight=0.07),
        'WebAttack' : dict(dur_mean=0.5,  pkt_mean=400,  fwd_pkts=6,  bwd_pkts=5,
                           fwd_bytes=3000, bwd_bytes=4000, syn=0.2, rst=0.1,
                           flow_bps=15000, weight=0.06),
        'BruteForce': dict(dur_mean=0.1,  pkt_mean=200,  fwd_pkts=4,  bwd_pkts=3,
                           fwd_bytes=800,  bwd_bytes=600,  syn=0.5, rst=0.05,
                           flow_bps=8000,  weight=0.06),
        'Botnet'    : dict(dur_mean=120.0, pkt_mean=500, fwd_pkts=20, bwd_pkts=18,
                           fwd_bytes=6000, bwd_bytes=5500, syn=0.1, rst=0.01,
                           flow_bps=500,   weight=0.04),
        'Exploit'   : dict(dur_mean=0.02, pkt_mean=1200, fwd_pkts=8,  bwd_pkts=6,
                           fwd_bytes=9000, bwd_bytes=7000, syn=0.3, rst=0.2,
                           flow_bps=200000, weight=0.02),
    }

    rows = []
    for threat_cls, prof in CLASS_PROFILES.items():
        n = int(n_samples * prof['weight'])
        w = prof

        dur        = np.abs(rng.normal(w['dur_mean'],   w['dur_mean']*0.5,  n))
        fwd_pkts   = np.abs(rng.poisson(w['fwd_pkts'],                      n)).astype(int) + 1
        bwd_pkts   = np.abs(rng.poisson(max(w['bwd_pkts'], 0.1),            n)).astype(int)
        tot_pkts   = fwd_pkts + bwd_pkts
        fwd_bytes  = np.abs(rng.normal(w['fwd_bytes'],  w['fwd_bytes']*0.3, n)).astype(int)
        bwd_bytes  = np.abs(rng.normal(w['bwd_bytes'],  w['bwd_bytes']*0.3, n)).astype(int)
        tot_bytes  = fwd_bytes + bwd_bytes

        # Derived features
        flow_duration_us = (dur * 1e6).astype(int) + 1
        flow_bps   = (tot_bytes * 8) / (dur + 1e-9)
        flow_pps   = tot_pkts / (dur + 1e-9)
        pkt_len_mean = tot_bytes / (tot_pkts + 1e-9)
        pkt_len_std  = pkt_len_mean * rng.uniform(0.1, 0.5, n)
        pkt_len_min  = np.maximum(20, pkt_len_mean - pkt_len_std * 2)
        pkt_len_max  = pkt_len_mean + pkt_len_std * 3

        # IAT features
        iat_mean   = dur / (tot_pkts + 1e-9)
        iat_std    = iat_mean * rng.uniform(0.05, 0.8, n)

        # Flag features
        syn_flag   = (rng.random(n) < w['syn']).astype(int)
        rst_flag   = (rng.random(n) < w['rst']).astype(int)
        ack_flag   = (rng.random(n) < 0.9).astype(int)
        urg_flag   = (rng.random(n) < 0.01).astype(int)
        fin_flag   = (rng.random(n) < 0.3).astype(int)
        psh_flag   = (rng.random(n) < 0.5).astype(int)

        # Header length
        fwd_hdr_len = rng.integers(20, 60, n)
        bwd_hdr_len = rng.integers(20, 60, n)

        # Window sizes
        init_win_fwd = rng.integers(64, 65535, n)
        init_win_bwd = rng.integers(64, 65535, n)

        for i in range(n):
            rows.append({
                # Flow identification
                'Flow Duration'              : flow_duration_us[i],

                # Packet counts
                'Total Fwd Packets'          : fwd_pkts[i],
                'Total Backward Packets'     : bwd_pkts[i],
                'Total Length of Fwd Packets': fwd_bytes[i],
                'Total Length of Bwd Packets': bwd_bytes[i],

                # Packet length stats
                'Fwd Packet Length Max'      : pkt_len_max[i],
                'Fwd Packet Length Min'      : pkt_len_min[i],
                'Fwd Packet Length Mean'     : pkt_len_mean[i],
                'Fwd Packet Length Std'      : pkt_len_std[i],
                'Bwd Packet Length Max'      : pkt_len_max[i] * 0.8,
                'Bwd Packet Length Min'      : pkt_len_min[i] * 0.8,
                'Bwd Packet Length Mean'     : pkt_len_mean[i] * 0.7,
                'Bwd Packet Length Std'      : pkt_len_std[i] * 0.6,

                # Flow rates
                'Flow Bytes/s'               : flow_bps[i],
                'Flow Packets/s'             : flow_pps[i],

                # IAT features
                'Flow IAT Mean'              : iat_mean[i],
                'Flow IAT Std'               : iat_std[i],
                'Flow IAT Max'               : iat_mean[i] + iat_std[i] * 3,
                'Flow IAT Min'               : max(0, iat_mean[i] - iat_std[i]),
                'Fwd IAT Total'              : dur[i] * 0.6,
                'Fwd IAT Mean'               : iat_mean[i] * 0.6,
                'Bwd IAT Total'              : dur[i] * 0.4,
                'Bwd IAT Mean'               : iat_mean[i] * 0.4,

                # Flags
                'SYN Flag Count'             : syn_flag[i],
                'RST Flag Count'             : rst_flag[i],
                'ACK Flag Count'             : ack_flag[i],
                'URG Flag Count'             : urg_flag[i],
                'FIN Flag Count'             : fin_flag[i],
                'PSH Flag Count'             : psh_flag[i],

                # Header lengths
                'Fwd Header Length'          : fwd_hdr_len[i],
                'Bwd Header Length'          : bwd_hdr_len[i],

                # Window sizes
                'Init_Win_bytes_forward'     : init_win_fwd[i],
                'Init_Win_bytes_backward'    : init_win_bwd[i],

                # Active/Idle
                'Active Mean'                : iat_mean[i] * 2,
                'Active Std'                 : iat_std[i],
                'Idle Mean'                  : iat_mean[i] * 3,
                'Idle Std'                   : iat_std[i] * 1.5,

                # Subflow
                'Subflow Fwd Packets'        : max(1, fwd_pkts[i] // 2),
                'Subflow Bwd Packets'        : max(0, bwd_pkts[i] // 2),
                'Subflow Fwd Bytes'          : fwd_bytes[i] // 2,
                'Subflow Bwd Bytes'          : bwd_bytes[i] // 2,

                # Labels
                'label'                      : threat_cls,
                'threat_class'               : threat_cls,
                'dataset'                    : 'SYNTHETIC',
                'source_file'                : 'synthetic_demo',
            })

    df = pd.DataFrame(rows)
    # Add small noise to numeric columns
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    for col in num_cols:
        noise = rng.normal(0, df[col].std() * 0.01, len(df))
        df[col] = (df[col] + noise).clip(lower=0)

    logger.info("Synthetic dataset generated: %d rows, %d features", *df.shape)
    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)


print("✅ Synthetic data generator defined.")

---
## 🚀 CELL 6 — Load Datasets (Real or Synthetic Fallback)

In [ ]:
# ============================================================
# IDRS — Master Data Load
# Tries real datasets first, falls back to synthetic
# ============================================================

print("🔍 Detecting available datasets...\n")

cic_files  = list(CIC_DIR.glob("*.csv"))
unsw_files = list(UNSW_DIR.glob("*.csv"))

print(f"   CIC-IDS2017  : {len(cic_files)} file(s) found in {CIC_DIR}")
print(f"   UNSW-NB15    : {len(unsw_files)} file(s) found in {UNSW_DIR}")

# ── Load CIC-IDS2017 ─────────────────────────────────────
df_cic = load_cicids2017(CIC_DIR, sample_frac=0.3, max_rows_per_file=150_000)

# ── Load UNSW-NB15 ───────────────────────────────────────
df_unsw = load_unsw_nb15(UNSW_DIR, sample_frac=0.5, max_rows_per_file=150_000)

# ── Fallback to synthetic if neither found ───────────────
USE_SYNTHETIC = (df_cic is None and df_unsw is None)

if USE_SYNTHETIC:
    print("\n⚠️  No real datasets found. Generating synthetic demo data...")
    df_main = generate_synthetic_idrs_data(n_samples=80_000)
    DATA_SOURCE = "SYNTHETIC"
elif df_cic is not None and df_unsw is None:
    df_main = df_cic
    DATA_SOURCE = "CIC-IDS2017"
elif df_cic is None and df_unsw is not None:
    df_main = df_unsw
    DATA_SOURCE = "UNSW-NB15"
else:
    # Both available — use CIC as primary (larger, more varied)
    df_main = df_cic
    DATA_SOURCE = "CIC-IDS2017+UNSW-NB15"
    print("\n✅ Both datasets loaded. Using CIC-IDS2017 as primary for Part 1 EDA.")
    print("   UNSW-NB15 will be merged in Part 2 preprocessing.")

print(f"\n{'='*55}")
print(f"  DATA SOURCE  : {DATA_SOURCE}")
print(f"  Total rows   : {len(df_main):,}")
print(f"  Total columns: {df_main.shape[1]}")
print(f"  Memory usage : {df_main.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"{'='*55}")

# Save metadata
meta = {
    'data_source'  : DATA_SOURCE,
    'n_rows'       : len(df_main),
    'n_columns'    : df_main.shape[1],
    'loaded_at'    : datetime.now().isoformat(),
    'threat_classes': df_main['threat_class'].value_counts().to_dict()
}
with open(OUTPUT_DIR / 'data_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

df_main.head(3)

---
## 🧹 CELL 7 — Data Quality Audit

In [ ]:
# ============================================================
# IDRS — Comprehensive Data Quality Audit
# ============================================================

def run_data_quality_audit(df: pd.DataFrame, name: str = "Dataset") -> pd.DataFrame:
    """Run a full data quality audit and return a summary DataFrame."""
    print(f"\n{'='*60}")
    print(f"  DATA QUALITY AUDIT — {name}")
    print(f"{'='*60}")

    num_df  = df.select_dtypes(include=np.number)
    cat_df  = df.select_dtypes(include='object')

    audit = []
    for col in df.columns:
        series = df[col]
        is_num = pd.api.types.is_numeric_dtype(series)
        rec = {
            'column'         : col,
            'dtype'          : str(series.dtype),
            'n_missing'      : series.isna().sum(),
            'pct_missing'    : round(series.isna().mean() * 100, 2),
            'n_unique'       : series.nunique(),
            'n_infinite'     : int(np.isinf(series).sum()) if is_num else 0,
            'n_negative'     : int((series < 0).sum()) if is_num else 0,
            'n_zero'         : int((series == 0).sum()) if is_num else 0,
            'mean'           : round(series.mean(), 4) if is_num else None,
            'std'            : round(series.std(), 4)  if is_num else None,
            'min'            : round(series.min(), 4)  if is_num else None,
            'max'            : round(series.max(), 4)  if is_num else None,
            'skewness'       : round(series.skew(), 4) if is_num else None,
        }
        audit.append(rec)

    audit_df = pd.DataFrame(audit)

    # ── Summary statistics ───────────────────────────────
    n_miss_cols  = (audit_df['n_missing'] > 0).sum()
    n_inf_cols   = (audit_df['n_infinite'] > 0).sum()
    n_neg_cols   = (audit_df['n_negative'] > 0).sum()
    n_dup        = df.duplicated().sum()
    n_const_cols = (audit_df['n_unique'] == 1).sum()

    print(f"  Rows          : {len(df):>10,}")
    print(f"  Columns       : {df.shape[1]:>10}")
    print(f"  Missing values: {df.isna().sum().sum():>10,} across {n_miss_cols} columns")
    print(f"  Infinite values:{int(np.isinf(df.select_dtypes(include=np.number)).sum().sum()):>9} across {n_inf_cols} columns")
    print(f"  Negative values:{n_neg_cols:>9} columns with negative flow features")
    print(f"  Duplicates    : {n_dup:>10,} rows")
    print(f"  Constant cols : {n_const_cols:>10} (zero-variance features)")

    # Flag problematic columns
    problems = audit_df[
        (audit_df['pct_missing'] > 5) |
        (audit_df['n_infinite'] > 0)  |
        (audit_df['n_unique']   == 1)
    ]
    if len(problems):
        print(f"\n  ⚠️  {len(problems)} columns need attention:")
        print(problems[['column','pct_missing','n_infinite','n_unique']].to_string(index=False))
    else:
        print("\n  ✅ No critical data quality issues detected.")

    return audit_df


# ── Run audit ────────────────────────────────────────────
audit_df = run_data_quality_audit(df_main, name=DATA_SOURCE)

# ── Apply fixes ──────────────────────────────────────────
print("\n🔧 Applying data quality fixes...")

df_clean = df_main.copy()
num_cols  = df_clean.select_dtypes(include=np.number).columns.tolist()

# 1. Replace infinities with NaN then median-fill
n_inf = np.isinf(df_clean[num_cols]).sum().sum()
df_clean[num_cols] = df_clean[num_cols].replace([np.inf, -np.inf], np.nan)
print(f"   ✅ Replaced {n_inf:,} infinite values with NaN")

# 2. Clip extreme negatives in flow features to 0
neg_flow_cols = [c for c in num_cols
                 if any(k in c.lower() for k in ['bytes','packets','length','duration'])]
df_clean[neg_flow_cols] = df_clean[neg_flow_cols].clip(lower=0)
print(f"   ✅ Clipped {len(neg_flow_cols)} flow feature columns to ≥ 0")

# 3. Median imputation for remaining NaNs
before_na = df_clean[num_cols].isna().sum().sum()
medians = df_clean[num_cols].median()
df_clean[num_cols] = df_clean[num_cols].fillna(medians)
print(f"   ✅ Median-imputed {before_na:,} NaN values")

# 4. Drop duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=num_cols[:10]).reset_index(drop=True)
print(f"   ✅ Removed {before - len(df_clean):,} duplicate rows → {len(df_clean):,} remain")

# 5. Drop zero-variance columns
zero_var = [c for c in num_cols if df_clean[c].std() < 1e-8]
if zero_var:
    df_clean = df_clean.drop(columns=zero_var)
    print(f"   ✅ Removed {len(zero_var)} zero-variance columns: {zero_var}")
else:
    print("   ✅ No zero-variance columns found")

print(f"\n📊 Clean dataset: {len(df_clean):,} rows × {df_clean.shape[1]} columns")

# Save cleaned dataset info
audit_df.to_csv(OUTPUT_DIR / 'data_quality_audit.csv', index=False)
print(f"💾 Audit saved to outputs/data_quality_audit.csv")

---
## 📊 CELL 8 — EDA: Class Distribution Analysis

In [ ]:
# ============================================================
# IDRS EDA — Class Distribution & Imbalance Analysis
# ============================================================

class_counts = df_clean['threat_class'].value_counts()
class_pct    = (class_counts / len(df_clean) * 100).round(2)

print("\n📊 Class Distribution:")
print(f"{'Class':<15} {'Count':>9} {'Percent':>9}")
print("-" * 36)
for cls in class_counts.index:
    print(f"{cls:<15} {class_counts[cls]:>9,} {class_pct[cls]:>8.2f}%")

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\n⚠️  Imbalance ratio (max/min): {imbalance_ratio:.1f}x")
if imbalance_ratio > 10:
    print("   → Severe imbalance detected. SMOTE will be applied in Part 2.")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Bar chart ────────────────────────────────────────────
ax = axes[0]
colors = [THREAT_TAXONOMY.get(c, {}).get('color', '#95a5a6') for c in class_counts.index]
bars = ax.bar(class_counts.index, class_counts.values, color=colors, edgecolor='white',
              linewidth=0.8, zorder=3)
ax.set_yscale('log')
ax.set_title('Threat Class Distribution (log scale)', fontweight='bold')
ax.set_xlabel('Threat Class')
ax.set_ylabel('Count (log)')
ax.tick_params(axis='x', rotation=30)
for bar, count in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1,
            f'{count:,}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# ── Pie chart ────────────────────────────────────────────
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    class_counts.values,
    labels     = class_counts.index,
    colors     = colors,
    autopct    = lambda p: f'{p:.1f}%' if p > 1.5 else '',
    startangle = 90,
    pctdistance= 0.82,
    wedgeprops = dict(linewidth=1.5, edgecolor='white'),
)
for t in autotexts:
    t.set_fontsize(8)
    t.set_fontweight('bold')
ax.set_title('Proportional Class Share', fontweight='bold')

plt.suptitle(f'IDRS Dataset — Class Distribution  |  Source: {DATA_SOURCE}  |  n={len(df_clean):,}',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"\n💾 Saved: outputs/eda_class_distribution.png")

---
## 📊 CELL 9 — EDA: Feature Statistical Profiles per Class

In [ ]:
# ============================================================
# IDRS EDA — Per-Class Feature Box Plots (Key Features)
# ============================================================

# Select the most discriminative features for visualization
KEY_FEATURES = [
    'Flow Duration', 'Flow Bytes/s', 'Flow Packets/s',
    'Total Fwd Packets', 'Total Backward Packets',
    'SYN Flag Count', 'RST Flag Count',
    'Fwd Packet Length Mean',
]

# Use only features that exist in the loaded dataframe
available_key = [f for f in KEY_FEATURES if f in df_clean.columns]
if not available_key:
    # Fallback: pick first 8 numeric columns
    available_key = df_clean.select_dtypes(include=np.number).columns[:8].tolist()

print(f"Plotting per-class distributions for: {available_key}\n")

n_feats = len(available_key)
n_cols  = 4
n_rows  = (n_feats + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4.5))
axes = axes.flatten()

classes     = sorted(df_clean['threat_class'].unique())
class_colors = [THREAT_TAXONOMY.get(c, {}).get('color', '#95a5a6') for c in classes]

for idx, feat in enumerate(available_key):
    ax = axes[idx]

    plot_data = [
        np.log1p(df_clean.loc[df_clean['threat_class'] == cls, feat].dropna().values)
        for cls in classes
    ]

    bp = ax.boxplot(
        plot_data,
        notch      = True,
        patch_artist= True,
        showfliers = False,
        widths     = 0.55,
    )
    for patch, color in zip(bp['boxes'], class_colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
    for median in bp['medians']:
        median.set(color='black', linewidth=2)

    ax.set_xticklabels(classes, rotation=35, ha='right', fontsize=8)
    ax.set_title(f'log({feat})', fontweight='bold', fontsize=9)
    ax.set_ylabel('log1p(value)', fontsize=8)

# Hide unused subplots
for idx in range(n_feats, len(axes)):
    axes[idx].set_visible(False)

# Legend
patches = [mpatches.Patch(color=THREAT_TAXONOMY.get(c, {}).get('color', '#95a5a6'),
                           label=c, alpha=0.8) for c in classes]
fig.legend(handles=patches, loc='lower center', ncol=len(classes),
           bbox_to_anchor=(0.5, -0.02), fontsize=9, frameon=True)

plt.suptitle('Feature Distributions per Threat Class (log-scaled, outliers hidden)',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.savefig(OUTPUT_DIR / 'eda_feature_boxplots.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"💾 Saved: outputs/eda_feature_boxplots.png")

---
## 📊 CELL 10 — EDA: Correlation Heatmap

In [ ]:
# ============================================================
# IDRS EDA — Correlation Heatmap (Top Features by Variance)
# ============================================================

num_df   = df_clean.select_dtypes(include=np.number)

# Select top 25 features by variance (most informative for visualization)
top_var_cols = num_df.var().nlargest(25).index.tolist()
corr_matrix  = num_df[top_var_cols].corr()

# Cluster the correlation matrix for better interpretability
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

dist_matrix  = 1 - np.abs(corr_matrix.values)
np.fill_diagonal(dist_matrix, 0)
dist_matrix  = np.clip(dist_matrix, 0, None)

try:
    linkage_matrix = linkage(squareform(dist_matrix), method='ward')
    ordered_idx    = leaves_list(linkage_matrix)
    ordered_cols   = [top_var_cols[i] for i in ordered_idx]
    corr_ordered   = corr_matrix.loc[ordered_cols, ordered_cols]
except Exception:
    corr_ordered   = corr_matrix  # fallback if clustering fails

fig, ax = plt.subplots(figsize=(14, 12))
mask     = np.triu(np.ones_like(corr_ordered, dtype=bool), k=1)

sns.heatmap(
    corr_ordered,
    ax          = ax,
    mask        = mask,
    cmap        = 'RdYlBu_r',
    center      = 0,
    vmin        = -1, vmax = 1,
    annot       = True,
    fmt         = '.2f',
    annot_kws   = {'size': 6},
    square      = True,
    linewidths  = 0.3,
    cbar_kws    = {'shrink': 0.75, 'label': 'Pearson Correlation'},
)

ax.set_title('Feature Correlation Heatmap\n(Top 25 features by variance, hierarchically clustered)',
             fontsize=12, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

# Identify highly correlated pairs (|r| > 0.95) — candidates for removal
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.95:
            high_corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r, 4)))

print(f"\n🔎 Highly correlated pairs (|r| > 0.95): {len(high_corr_pairs)}")
for a, b, r in high_corr_pairs[:10]:
    print(f"   {a:35s} ↔ {b:35s}  r={r}")
if len(high_corr_pairs) > 10:
    print(f"   ... and {len(high_corr_pairs)-10} more (will be pruned in Part 2)")

HIGH_CORR_PAIRS = high_corr_pairs
print(f"\n💾 Saved: outputs/eda_correlation_heatmap.png")

---
## 📊 CELL 11 — EDA: Attack Pattern Temporal Analysis

In [ ]:
# ============================================================
# IDRS EDA — Temporal & Source File Attack Pattern Analysis
# ============================================================

if 'source_file' in df_clean.columns and df_clean['source_file'].nunique() > 1:
    # CIC-IDS2017 has day-based files: analyze day-by-day attack profiles
    day_attack = df_clean.groupby(['source_file', 'threat_class']).size().unstack(fill_value=0)
    day_attack_pct = day_attack.div(day_attack.sum(axis=1), axis=0) * 100

    fig, axes = plt.subplots(2, 1, figsize=(16, 12))

    # ── Stacked bar: absolute counts ──────────────────────
    ax = axes[0]
    colors = [THREAT_TAXONOMY.get(c, {}).get('color', '#95a5a6') for c in day_attack.columns]
    day_attack.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.75, edgecolor='white')
    ax.set_title('Daily Attack Volume by Threat Class (CIC-IDS2017)', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Flow Count')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
    ax.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, p: f'{x:,.0f}'))

    # ── Stacked bar: percentage composition ───────────────
    ax = axes[1]
    day_attack_pct.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.75, edgecolor='white')
    ax.set_title('Daily Attack Composition (%) by Threat Class', fontweight='bold')
    ax.set_xlabel('Day / Source File')
    ax.set_ylabel('Percentage of Flows')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
    ax.set_ylim(0, 105)

    plt.suptitle('Temporal Attack Distribution — Educational Platform Traffic',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'eda_temporal_attack.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f"💾 Saved: outputs/eda_temporal_attack.png")

else:
    # Synthetic / single-file: use simulated hourly distribution
    print("ℹ️  Single-file dataset detected. Simulating hourly attack timeline...")

    hours = np.arange(24)
    # Educational platform: attacks peak during exam hours (8-10am, 2-4pm) and off-hours
    attack_intensity = np.array([
        0.3, 0.2, 0.2, 0.3, 0.4, 0.6, 1.2, 2.5,   # 0-7
        4.0, 5.5, 3.2, 2.1, 3.0, 3.5, 4.2, 4.8,   # 8-15
        3.1, 2.5, 2.0, 1.5, 1.2, 0.9, 0.7, 0.5    # 16-23
    ])

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.fill_between(hours, attack_intensity, alpha=0.35, color='#e74c3c', label='Attack Intensity')
    ax.plot(hours, attack_intensity, '-o', color='#c0392b', linewidth=2, markersize=5)

    benign_intensity = 5.0 - attack_intensity * 0.3
    ax.fill_between(hours, benign_intensity, alpha=0.25, color='#2ecc71', label='Benign Traffic')
    ax.plot(hours, benign_intensity, '-s', color='#27ae60', linewidth=2, markersize=5)

    # Annotate exam periods
    for hr, label in [(8, 'Morning\nExams'), (14, 'Afternoon\nExams')]:
        ax.axvspan(hr, hr+2, alpha=0.12, color='#f39c12')
        ax.text(hr+1, max(attack_intensity)*1.05, label, ha='center', fontsize=8,
                color='#e67e22', fontweight='bold')

    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Relative Traffic Intensity')
    ax.set_title('Simulated Hourly Attack Pattern — University Web Platform\n'
                 '(Orange: exam/peak hours with elevated attack risk)',
                 fontweight='bold')
    ax.set_xticks(hours)
    ax.set_xticklabels([f'{h:02d}:00' for h in hours], rotation=45, fontsize=8)
    ax.legend(fontsize=10)
    ax.set_xlim(0, 23)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'eda_temporal_attack.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f"💾 Saved: outputs/eda_temporal_attack.png")

---
## 🔬 CELL 12 — Feature Engineering

In [ ]:
# ============================================================
# IDRS — Feature Engineering
# Derives security-relevant features from raw flow data
# ============================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Engineer domain-specific security features from raw network flows.
    All features are interpretable and grounded in network security literature.
    """
    df = df.copy()
    eps = 1e-9  # prevent division by zero

    print("🔬 Engineering features...")

    # ── 1. Byte Asymmetry Ratio ───────────────────────────
    # High ratio (>10) indicates potential data exfiltration or DoS response
    if 'Total Length of Fwd Packets' in df.columns and 'Total Length of Bwd Packets' in df.columns:
        fwd_b = df['Total Length of Fwd Packets']
        bwd_b = df['Total Length of Bwd Packets']
        df['feat_byte_asymmetry']  = (fwd_b - bwd_b) / (fwd_b + bwd_b + eps)
        df['feat_bwd_fwd_ratio']   = bwd_b / (fwd_b + eps)
        print("   ✅ Byte asymmetry features")

    # ── 2. Packet Rate Entropy ────────────────────────────
    # Uniform distributions (entropy≈0) suggest automated/scripted attacks
    if 'Flow Bytes/s' in df.columns and 'Flow Packets/s' in df.columns:
        flow_bps = df['Flow Bytes/s'].clip(lower=0)
        flow_pps = df['Flow Packets/s'].clip(lower=0)
        # Log-ratio of bytes-per-packet (large = large payload, small = header-only flood)
        df['feat_bytes_per_packet']= flow_bps / (flow_pps + eps)
        print("   ✅ Bytes-per-packet ratio")

    # ── 3. SYN/ACK Ratio (TCP Handshake Completeness) ────
    # SYN floods: SYN=1, ACK=0; completed sessions: SYN=ACK
    if 'SYN Flag Count' in df.columns and 'ACK Flag Count' in df.columns:
        syn = df['SYN Flag Count']
        ack = df['ACK Flag Count']
        df['feat_syn_ack_ratio']   = syn / (ack + eps)
        df['feat_handshake_complete'] = ((syn > 0) & (ack > 0)).astype(int)
        print("   ✅ SYN/ACK handshake features")

    # ── 4. RST Anomaly Score ──────────────────────────────
    # Elevated RST rates indicate port scanning or abrupt connection terminations
    if 'RST Flag Count' in df.columns:
        if 'Total Fwd Packets' in df.columns:
            df['feat_rst_pkt_ratio'] = df['RST Flag Count'] / (df['Total Fwd Packets'] + eps)
        print("   ✅ RST anomaly score")

    # ── 5. Payload Variance (evasion detection) ───────────
    # Attackers often send uniform payload sizes; legit traffic varies
    if 'Fwd Packet Length Std' in df.columns and 'Fwd Packet Length Mean' in df.columns:
        fwd_std  = df['Fwd Packet Length Std']
        fwd_mean = df['Fwd Packet Length Mean']
        # Coefficient of variation: low CoV = uniform payload = suspicious
        df['feat_payload_cov']     = fwd_std / (fwd_mean + eps)
        print("   ✅ Payload coefficient of variation")

    # ── 6. Flow Duration Bucket ──────────────────────────
    # Ultra-short flows (<1ms) = scanning; ultra-long (>1min) = persistence/botnet
    if 'Flow Duration' in df.columns:
        dur = df['Flow Duration']  # microseconds in CIC-IDS2017
        df['feat_dur_log']         = np.log1p(dur)
        df['feat_is_ultrashort']   = (dur < 1000).astype(int)      # <1ms
        df['feat_is_ultralong']    = (dur > 60_000_000).astype(int) # >1min
        print("   ✅ Flow duration features")

    # ── 7. Header/Payload Ratio ──────────────────────────
    # Probe/scan packets: mostly headers, minimal payload
    if 'Fwd Header Length' in df.columns and 'Total Length of Fwd Packets' in df.columns:
        hdr = df['Fwd Header Length']
        pld = df['Total Length of Fwd Packets']
        df['feat_header_payload_ratio'] = hdr / (pld + eps)
        print("   ✅ Header-to-payload ratio")

    # ── 8. Bi-directional Symmetry Score ─────────────────
    # Legitimate traffic: symmetric; DDoS: highly asymmetric
    if 'Total Fwd Packets' in df.columns and 'Total Backward Packets' in df.columns:
        fwd_p = df['Total Fwd Packets']
        bwd_p = df['Total Backward Packets']
        df['feat_pkt_symmetry']    = 1 - abs(fwd_p - bwd_p) / (fwd_p + bwd_p + eps)
        df['feat_no_response']     = ((bwd_p == 0) & (fwd_p > 0)).astype(int)
        print("   ✅ Bidirectional symmetry + no-response flag")

    # ── 9. Interarrival Time Burstiness ──────────────────
    # Bursty = high std/mean IAT; automated floods: very regular IAT
    if 'Flow IAT Std' in df.columns and 'Flow IAT Mean' in df.columns:
        iat_mean = df['Flow IAT Mean'].clip(lower=0)
        iat_std  = df['Flow IAT Std'].clip(lower=0)
        df['feat_iat_burstiness']  = iat_std / (iat_mean + eps)
        print("   ✅ IAT burstiness")

    # ── 10. URG Flag Anomaly ──────────────────────────────
    # URG flags in bulk = port scanner or exploit kit
    if 'URG Flag Count' in df.columns:
        df['feat_urg_present']     = (df['URG Flag Count'] > 0).astype(int)
        print("   ✅ URG flag anomaly")

    # ── Count new features ───────────────────────────────
    new_feats = [c for c in df.columns if c.startswith('feat_')]
    print(f"\n   📐 Total engineered features: {len(new_feats)}")
    print(f"   {new_feats}")

    return df


# Apply feature engineering
df_featured = engineer_features(df_clean)

engineered_cols = [c for c in df_featured.columns if c.startswith('feat_')]
print(f"\n📊 Dataset with engineered features: {df_featured.shape}")

---
## 📊 CELL 13 — EDA: Engineered Feature Distributions

In [ ]:
# ============================================================
# IDRS EDA — Engineered Feature Violin Plots
# ============================================================

viz_feats = [c for c in engineered_cols if c in df_featured.columns][:8]

if not viz_feats:
    print("ℹ️  No engineered features found — skipping violin plot.")
else:
    n_cols = 4
    n_rows = (len(viz_feats) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 5))
    axes = np.array(axes).flatten()

    classes = sorted(df_featured['threat_class'].unique())

    for idx, feat in enumerate(viz_feats):
        ax = axes[idx]
        plot_data = []
        positions = []
        c_colors  = []

        for k, cls in enumerate(classes):
            vals = df_featured.loc[df_featured['threat_class'] == cls, feat].dropna()
            vals = np.clip(vals, vals.quantile(0.01), vals.quantile(0.99))
            if len(vals) > 10:
                plot_data.append(vals.values)
                positions.append(k)
                c_colors.append(THREAT_TAXONOMY.get(cls, {}).get('color', '#95a5a6'))

        if plot_data:
            parts = ax.violinplot(plot_data, positions=positions, showmedians=True,
                                  showextrema=True, widths=0.7)
            for pc, color in zip(parts['bodies'], c_colors):
                pc.set_facecolor(color)
                pc.set_alpha(0.75)
            parts['cmedians'].set_color('black')
            parts['cmedians'].set_linewidth(2)

        ax.set_xticks(range(len(classes)))
        ax.set_xticklabels(classes, rotation=35, ha='right', fontsize=8)
        ax.set_title(feat.replace('feat_', '').replace('_', ' ').title(),
                     fontsize=9, fontweight='bold')

    for idx in range(len(viz_feats), len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('Engineered Feature Distributions per Threat Class',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'eda_engineered_features.png', bbox_inches='tight', dpi=150)
    plt.show()
    print(f"💾 Saved: outputs/eda_engineered_features.png")

---
## 📊 CELL 14 — EDA: Class-wise Feature Statistics Table

In [ ]:
# ============================================================
# IDRS EDA — Per-Class Feature Statistics
# ============================================================

print("\n📋 Per-Class Feature Statistics (mean ± std)")
print("="*80)

stat_feats = [
    f for f in (available_key + engineered_cols)
    if f in df_featured.columns
][:12]

stats_table = df_featured.groupby('threat_class')[stat_feats].agg(['mean','std'])

# Flatten multi-level columns
stats_table.columns = [f"{col}_{stat}" for col, stat in stats_table.columns]

# Display mean columns only for readability
mean_cols = [c for c in stats_table.columns if c.endswith('_mean')]
display_df = stats_table[mean_cols].copy()
display_df.columns = [c.replace('_mean','') for c in mean_cols]

# Format nicely
pd.set_option('display.float_format', '{:.3f}'.format)
print(display_df.to_string())
pd.set_option('display.float_format', '{:.4f}'.format)

# Save
display_df.to_csv(OUTPUT_DIR / 'eda_class_feature_stats.csv')
print(f"\n💾 Saved: outputs/eda_class_feature_stats.csv")

# ── Statistical significance tests ────────────────────────
print("\n📊 Kruskal-Wallis Test (feature discriminability across classes):")
print(f"{'Feature':<40} {'H-stat':>10} {'p-value':>12} {'Significant':>12}")
print("-" * 78)

kw_results = []
for feat in stat_feats:
    if feat not in df_featured.columns:
        continue
    groups = [
        df_featured.loc[df_featured['threat_class'] == cls, feat].dropna().values
        for cls in df_featured['threat_class'].unique()
        if len(df_featured.loc[df_featured['threat_class'] == cls, feat].dropna()) > 2
    ]
    if len(groups) >= 2:
        try:
            h_stat, p_val = stats.kruskal(*groups)
            sig = "✅ YES" if p_val < 0.05 else "❌ NO"
            print(f"  {feat:<38} {h_stat:>10.2f} {p_val:>12.2e} {sig:>12}")
            kw_results.append({'feature': feat, 'H': h_stat, 'p': p_val, 'significant': p_val < 0.05})
        except Exception:
            pass

n_sig = sum(r['significant'] for r in kw_results)
print(f"\n  → {n_sig}/{len(kw_results)} features are statistically discriminative (p < 0.05)")

---
## 💾 CELL 15 — Save Processed Dataset & Part 1 Summary

In [ ]:
# ============================================================
# IDRS — Save Processed Dataset for Part 2
# ============================================================

# ── Select final feature set ─────────────────────────────
# Numeric flow features + engineered features + label columns
num_flow_cols    = df_featured.select_dtypes(include=np.number).columns.tolist()
label_cols       = ['label', 'threat_class', 'dataset', 'source_file']
label_cols       = [c for c in label_cols if c in df_featured.columns]

# Remove highly correlated duplicates (|r| > 0.95)
cols_to_drop = set()
for a, b, _ in HIGH_CORR_PAIRS:
    if b not in label_cols and b in num_flow_cols:
        cols_to_drop.add(b)

final_num_cols = [c for c in num_flow_cols if c not in cols_to_drop]
final_cols     = final_num_cols + label_cols
final_cols     = list(dict.fromkeys(final_cols))  # deduplicate preserving order

df_out = df_featured[[c for c in final_cols if c in df_featured.columns]].copy()

# ── Encode label for downstream use ─────────────────────
le = LabelEncoder()
df_out['label_encoded'] = le.fit_transform(df_out['threat_class'])

# Save label encoder classes
label_classes = {int(i): str(c) for i, c in enumerate(le.classes_)}
with open(OUTPUT_DIR / 'label_classes.json', 'w') as f:
    json.dump(label_classes, f, indent=2)

# ── Save processed dataset ────────────────────────────────
output_path = OUTPUT_DIR / 'idrs_processed_part1.parquet'
df_out.to_parquet(output_path, index=False, compression='snappy')

print("✅ Processed dataset saved.")
print(f"   Path   : {output_path}")
print(f"   Rows   : {len(df_out):,}")
print(f"   Columns: {df_out.shape[1]} ({len(final_num_cols)} numeric + {len(label_cols)+1} label)")
print(f"   Removed: {len(cols_to_drop)} highly correlated features")
print(f"   Classes: {label_classes}")

# ── Part 1 Summary Dashboard ──────────────────────────────
fig = plt.figure(figsize=(18, 8))
fig.patch.set_facecolor('#1a1a2e')

summary_text = [
    ("DATA SOURCE",           DATA_SOURCE),
    ("Total Flows",           f"{len(df_out):,}"),
    ("Numeric Features",      str(len(final_num_cols))),
    ("Engineered Features",   str(len(engineered_cols))),
    ("Threat Classes",        str(df_out['threat_class'].nunique())),
    ("Removed (high-corr)",  str(len(cols_to_drop))),
    ("Duplicate rows removed",str(len(df_main) - len(df_clean))),
    ("Sig. features (KW)",   f"{n_sig}/{len(kw_results)}"),
    ("Imbalance ratio",       f"{imbalance_ratio:.1f}x"),
    ("SMOTE needed",          "YES" if imbalance_ratio > 10 else "MILD"),
    ("Output format",         "Snappy Parquet"),
    ("Label encoder",         "outputs/label_classes.json"),
]

ax = fig.add_subplot(111)
ax.set_facecolor('#1a1a2e')
ax.axis('off')

ax.text(0.5, 0.97, '🛡️  IDRS PART 1 — SUMMARY',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=18, fontweight='bold', color='white')
ax.text(0.5, 0.90, 'Environment Setup, EDA & Feature Engineering Complete',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=11, color='#a0a0c0')

for i, (key, val) in enumerate(summary_text):
    col_offset = 0.05 if i < 6 else 0.52
    row = i % 6
    y_pos = 0.78 - row * 0.11
    ax.text(col_offset, y_pos, f"{key}:",
            transform=ax.transAxes, va='center',
            fontsize=10, color='#a0a0c0', fontweight='bold')
    ax.text(col_offset + 0.22, y_pos, val,
            transform=ax.transAxes, va='center',
            fontsize=10, color='#2ecc71', fontweight='bold')

ax.text(0.5, 0.05, '→ Part 2: Preprocessing Pipeline, SMOTE, Classical ML Baseline & Optuna Tuning',
        transform=ax.transAxes, ha='center', va='bottom',
        fontsize=9, color='#f39c12', style='italic')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'part1_summary.png', bbox_inches='tight',
            dpi=150, facecolor='#1a1a2e')
plt.show()

logger.info("Part 1 complete. Processed dataset: %s", output_path)
print("\n" + "="*60)
print("✅ PART 1 COMPLETE")
print("   Outputs:")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(f"   • {f.name}  ({f.stat().st_size/1024:.1f} KB)")
print("\n→ Proceed to IDRS_Part2_Preprocessing_ClassicalML.ipynb")
print("="*60)

---
## ✅ Part 1 — Completion Checklist

| Task | Status |
|---|---|
| Environment setup & dependency installation | ✅ |
| Directory structure creation | ✅ |
| Educational threat taxonomy definition | ✅ |
| Dataset loader (CIC-IDS2017 + UNSW-NB15) | ✅ |
| Synthetic fallback generator | ✅ |
| Data quality audit (nulls, infinities, duplicates) | ✅ |
| Automated data quality fixes | ✅ |
| Class distribution EDA | ✅ |
| Per-class feature distribution boxplots | ✅ |
| Correlation heatmap (hierarchically clustered) | ✅ |
| Temporal/hourly attack pattern analysis | ✅ |
| Domain-specific feature engineering (10 families) | ✅ |
| Kruskal-Wallis discriminability tests | ✅ |
| High-correlation pruning | ✅ |
| Processed dataset saved (Parquet) | ✅ |
| Label encoder saved (JSON) | ✅ |

---
### ➡️ Next: `IDRS_Part2_Preprocessing_ClassicalML.ipynb`
- Full preprocessing pipeline (StandardScaler, SMOTE)
- Random Forest + XGBoost + LightGBM baselines  
- Optuna Bayesian hyperparameter optimization  
- SHAP explainability  
- Comprehensive evaluation suite